# Seller-Portal Web App (Colab) — Live two-model e-vaporiser check

This notebook launches the **seller-portal web interface** as a public web server straight from Colab.
A visitor uploads a listing photo and the **two best models** from training score it **live** (online inference),
side-by-side, with a moderation verdict.

**Before you start** you need the `best_models/` folder produced by `evaporiser_classifier.ipynb`
(two `*.pt` checkpoints + `manifest.json`). You can either upload it or load it from Google Drive.

Run the cells top to bottom; the last cell prints a public **`*.gradio.live`** URL you can share.

## Step 0 — Install dependencies

In [16]:
%%capture
!pip install -q torch torchvision transformers gradio

## Step 1 — Provide the trained models

**Option A — Google Drive** (recommended if you copied `best_models/` to Drive in the training notebook):

In [17]:
from google.colab import drive
drive.mount('/content/drive')

# adjust if you saved it elsewhere
MODELS_DIR = '/content/drive/MyDrive/best_models'
import os
print('exists:', os.path.isdir(MODELS_DIR), '->', MODELS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
exists: True -> /content/drive/MyDrive/best_models


**Option B — Upload a zip** instead. Skip this cell if you used Option A.
Zip your folder first (`zip -r best_models.zip best_models`), then run this and pick the zip.

In [ ]:
# from google.colab import files
# up = files.upload()                       # choose best_models.zip
# import zipfile, io
# name = next(iter(up))
# with zipfile.ZipFile(io.BytesIO(up[name])) as z:
#     z.extractall('/content')
# MODELS_DIR = '/content/best_models'
# print('extracted to', MODELS_DIR)

## Step 2 — Define the model loading + seller-portal interface

This is the same code as the standalone `seller_portal_app.py`, so the local app and this Colab server behave
identically. It rebuilds each model from its checkpoint metadata and wires up the Gradio UI.

In [20]:
import os
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import resnet18
from PIL import Image
import gradio as gr

try:
    from transformers import ViTForImageClassification
except Exception:  # transformers only needed if a ViT checkpoint is loaded
    ViTForImageClassification = None


# --------------------------------------------------------------------------- model
class ViTWrapper(nn.Module):
    """Makes a HuggingFace ViT return plain logits, matching the training setup."""

    def __init__(self, vit):
        super().__init__()
        self.vit = vit

    def forward(self, x):
        return self.vit(pixel_values=x).logits


def build_model(backbone, num_classes, vit_checkpoint):
    """Recreate the architecture so the saved weights can be loaded into it."""
    if backbone == "resnet":
        net = resnet18(weights=None)
        net.fc = nn.Linear(net.fc.in_features, num_classes)
        return net
    if backbone == "vit":
        if ViTForImageClassification is None:
            raise RuntimeError("transformers is required to load the ViT model.")
        vit = ViTForImageClassification.from_pretrained(
            vit_checkpoint, num_labels=num_classes, ignore_mismatched_sizes=True
        )
        return ViTWrapper(vit)
    raise ValueError(f"unknown backbone: {backbone}")


class LoadedModel:
    """Holds a ready-to-use model plus its preprocessing and metadata."""

    def __init__(self, name, ckpt_path, device):
        self.name = name
        self.device = device
        ckpt = torch.load(ckpt_path, map_location=device)
        self.class_names = ckpt["class_names"]
        self.backbone = ckpt["backbone"]
        self.strategy = ckpt["strategy"]
        net = build_model(self.backbone, len(self.class_names),
                          ckpt.get("vit_checkpoint", "google/vit-base-patch16-224"))
        net.load_state_dict(ckpt["state_dict"])
        net.eval().to(device)
        self.net = net
        norm = ckpt["norm"]
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(ckpt.get("img_size", 224)),
            transforms.ToTensor(),
            transforms.Normalize(norm["mean"], norm["std"]),
        ])
        # index of the "vape" class (falls back to last class)
        self.vape_idx = (self.class_names.index("vape")
                         if "vape" in self.class_names else len(self.class_names) - 1)

    @torch.no_grad()
    def predict(self, pil_img):
        x = self.transform(pil_img.convert("RGB")).unsqueeze(0).to(self.device)
        probs = F.softmax(self.net(x), dim=1)[0].cpu().tolist()
        scores = {self.class_names[i]: float(probs[i]) for i in range(len(probs))}
        return scores, float(probs[self.vape_idx])

    @property
    def pretty_name(self):
        return f"{self.backbone.upper()} · {self.strategy.replace('_', ' ')}"


def load_models(models_dir, device):
    manifest_path = os.path.join(models_dir, "manifest.json")
    if os.path.exists(manifest_path):
        with open(manifest_path) as f:
            entries = json.load(f)
    else:  # no manifest: just grab any .pt files
        entries = [{"name": os.path.splitext(f)[0],
                    "path": os.path.join(models_dir, f)}
                   for f in sorted(os.listdir(models_dir)) if f.endswith(".pt")]
    models = []
    for e in entries[:2]:
        # Always look for the checkpoint inside models_dir first. The path stored
        # in manifest.json is absolute from the *training* environment
        # (e.g. /content/best_models/vit_full.pt) and won't exist here when the
        # models were copied elsewhere, e.g. to Google Drive. Fall back to the
        # stored path only if the local file is missing.
        local = os.path.join(models_dir, os.path.basename(e["path"]))
        path = local if os.path.exists(local) else e["path"]
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Checkpoint for '{e['name']}' not found. Looked in {local!r} "
                f"and {e['path']!r}. Set MODELS_DIR to the folder that contains "
                f"the .pt files.")
        models.append(LoadedModel(e["name"], path, device))
    if not models:
        raise FileNotFoundError(f"No model checkpoints found in {models_dir}")
    return models


# --------------------------------------------------------------------------- UI copy/logic
def verdict(vape_probs, threshold):
    """Combine the two models' vape probabilities into one moderation decision."""
    flags = [p >= threshold for p in vape_probs]
    avg = sum(vape_probs) / len(vape_probs)
    if all(flags):
        return ("blocked", "Blocked — looks like an e-vaporiser",
                "Both models flagged this photo. The listing cannot be published.")
    if any(flags):
        return ("review", "Needs review — possible e-vaporiser",
                "The models disagree. Sent to a human reviewer before publishing.")
    return ("allowed", "Looks fine — no e-vaporiser detected",
            "Both models cleared this photo. The listing can be published.")


# Intentional palette: a calm trust-&-safety ops console.
# ink #15233B (deep slate) · paper #F7F8FA · line #E2E6EC · brand #2D6CDF
# clear #1F9D6B · review #C9820B · block #C2453C
CSS = """
:root{
  --ink:#15233B; --muted:#5C6B82; --paper:#F7F8FA; --card:#FFFFFF;
  --line:#E2E6EC; --brand:#2D6CDF; --clear:#1F9D6B; --review:#C9820B; --block:#C2453C;
}
.gradio-container{background:var(--paper)!important; font-family:'Inter',system-ui,sans-serif;}
#hero{display:flex;align-items:center;gap:14px;padding:18px 22px;margin-bottom:6px;
  background:var(--ink);border-radius:14px;color:#fff;}
#hero .dot{width:34px;height:34px;border-radius:9px;background:var(--brand);
  display:flex;align-items:center;justify-content:center;font-size:18px;}
#hero h1{font-size:18px;margin:0;font-weight:650;letter-spacing:.2px;}
#hero p{margin:2px 0 0;font-size:12.5px;color:#AEBAD0;}
.panel{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:18px;}
.eyebrow{font-size:11px;font-weight:700;letter-spacing:.12em;text-transform:uppercase;
  color:var(--muted);margin-bottom:10px;}
.verdict{border-radius:12px;padding:16px 18px;margin-bottom:14px;border:1px solid transparent;}
.verdict h2{margin:0 0 4px;font-size:16px;color:inherit!important;}
.verdict p{margin:0;font-size:13px;opacity:.92;color:inherit!important;}
.verdict.allowed{background:#E9F7F1;border-color:#BFE7D6;color:#0E5C3F;}
.verdict.review {background:#FBF1DC;border-color:#F0DCA8;color:#7A5206;}
.verdict.blocked{background:#FAE9E7;border-color:#F0C5C0;color:#7E2A24;}
.modelcard{border:1px solid var(--line);border-radius:12px;padding:14px 16px;margin-bottom:12px;}
.modelcard .top{display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px;}
.modelcard .nm{font-weight:650;color:var(--ink);font-size:14px;}
.modelcard .acc{font-size:11.5px;color:var(--muted);}
.bar{height:9px;border-radius:6px;background:#EEF1F5;overflow:hidden;}
.bar>span{display:block;height:100%;}
.barrow{display:flex;justify-content:space-between;font-size:12px;color:var(--muted);margin:9px 0 4px;}
.barrow b{color:var(--ink);}
.hint{font-size:12px;color:var(--muted);line-height:1.5;}
footer{display:none!important;}
"""

HERO = """
<div id='hero'>
  <div class='dot'>🛡️</div>
  <div>
    <h1>Marketplace Listing Check</h1>
    <p>Prohibited-item screening · e-vaporiser detection · two-model comparison</p>
  </div>
</div>
"""


def build_interface(models):
    def render(img, threshold):
        if img is None:
            empty = "<div class='panel'><div class='eyebrow'>Listing review</div>" \
                    "<p class='hint'>Add a product photo to run the check.</p></div>"
            return empty

        results = [m.predict(img) for m in models]
        vape_probs = [r[1] for r in results]
        state, title, detail = verdict(vape_probs, threshold)

        cards = ""
        for m, (scores, vp) in zip(models, results):
            conf = max(scores.values())
            top = max(scores, key=scores.get)
            bar_color = "var(--block)" if vp >= threshold else "var(--clear)"
            cards += f"""
            <div class='modelcard'>
              <div class='top'>
                <span class='nm'>{m.pretty_name}</span>
                <span class='acc'>{m.name}</span>
              </div>
              <div class='barrow'><span>vape probability</span><b>{vp*100:.1f}%</b></div>
              <div class='bar'><span style='width:{vp*100:.1f}%;background:{bar_color}'></span></div>
              <div class='barrow' style='margin-top:8px;'>
                <span>top label</span><b>{top} · {conf*100:.1f}% confidence</b></div>
            </div>"""

        agree = "agree" if (vape_probs[0] >= threshold) == (vape_probs[1] >= threshold) else "disagree"
        return f"""
        <div class='panel'>
          <div class='eyebrow'>Listing review</div>
          <div class='verdict {state}'><h2>{title}</h2><p>{detail}</p></div>
          {cards}
          <p class='hint'>The two models <b style='color:var(--review);font-weight:700'>{agree}</b> at the current threshold
          ({threshold*100:.0f}%). Lower the threshold to catch more vapes
          (higher recall); raise it to reduce false alarms on honest listings.</p>
        </div>"""

    with gr.Blocks(css=CSS, title="Marketplace Listing Check") as demo:
        gr.HTML(HERO)
        with gr.Row():
            with gr.Column(scale=1):
                gr.HTML("<div class='eyebrow'>New listing</div>")
                gr.Textbox(label="Item title", placeholder="e.g. Vintage desk lamp",
                           value="")
                gr.Textbox(label="Price", placeholder="$0.00", value="")
                photo = gr.Image(type="pil", label="Product photo",
                                 sources=["upload", "clipboard"], height=300)
                threshold = gr.Slider(0.05, 0.95, value=0.5, step=0.05,
                                      label="Block threshold (vape probability)")
                gr.HTML("<p class='hint'>Trust & Safety policy: photos likely to show an "
                        "e-vaporiser are blocked before the listing goes live.</p>")
            with gr.Column(scale=1):
                review = gr.HTML("<div class='panel'><div class='eyebrow'>Listing review</div>"
                                 "<p class='hint'>Add a product photo to run the check.</p></div>")

        photo.change(render, [photo, threshold], review)
        threshold.change(render, [photo, threshold], review)

    return demo

## Step 3 — Launch the public web server

`share=True` gives a public link (live for ~72h). Both models run **online inference** on every uploaded image.

In [21]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Loading models on', device)

models = load_models(MODELS_DIR, device)
for m in models:
    print('  loaded', m.name, '->', m.pretty_name)

demo = build_interface(models)

# share=True needs Gradio's public tunnel server (frpc). It occasionally fails
# with "Could not create share link" / "session shutdown" when that server or
# the network is unreachable — this is infrastructure, not a bug in the app.
# Retry a few times, then fall back to an inline launch so the app is still
# usable inside this Colab tab (just without a public *.gradio.live URL).
launched = False
for attempt in range(3):
    try:
        demo.launch(share=True)
        launched = True
        break
    except Exception as e:
        print(f'  share link attempt {attempt + 1}/3 failed: {e}')
        try:
            demo.close()   # free the port before retrying
        except Exception:
            pass

if not launched:
    print('\nCould not create a public share link after 3 tries '
          '(Gradio tunnel unreachable — check https://status.gradio.app).')
    print('Launching inline instead: the app works in this Colab output cell, '
          'but there is no public URL to share.')
    demo.launch(share=False)

Loading models on cuda


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

...

  loaded vit_full -> VIT · full
  loaded vit_head -> VIT · head


/tmp/ipykernel_3419/2638939295.py:214: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="Marketplace Listing Check") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/07/01 03:07:11 [W] [service.go:132] login to server failed: session shutdown


<IPython.core.display.Javascript object>

In [ ]:
# ---------------------------------------------------------------------------
# ALTERNATIVE PUBLIC LINK — use ONLY if the cell above fails to create a
# gradio.live share link (e.g. "Could not create share link" / expired TLS
# certificate on Gradio's tunnel server). This routes a public URL through a
# free Cloudflare Quick Tunnel instead — no account or token required.
# ---------------------------------------------------------------------------
import os, re, stat, subprocess, atexit, urllib.request

PORT = 7860
CF = '/usr/local/bin/cloudflared'

# 1) Download the cloudflared binary once.
if not os.path.exists(CF):
    print('Downloading cloudflared...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        CF)
    os.chmod(CF, os.stat(CF).st_mode | stat.S_IEXEC)

# 2) (Re)launch the app locally, non-blocking, on a fixed port.
try:
    demo.close()          # close any server started by the cell above
except Exception:
    pass
demo = build_interface(models)
demo.launch(server_port=PORT, share=False, prevent_thread_lock=True)

# 3) Open the Cloudflare tunnel and print the public URL it hands back.
proc = subprocess.Popen([CF, 'tunnel', '--url', f'http://localhost:{PORT}',
                         '--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
atexit.register(proc.terminate)     # clean up the tunnel when the kernel stops

public_url = None
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[-\w.]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

print('\n' + '=' * 60)
print('PUBLIC URL (share this):', public_url or 'not found — see log above')
print('=' * 60)

---
**Troubleshooting**
- *`No model checkpoints found`* — check `MODELS_DIR` points at the folder containing the `.pt` files.
- *ViT fails to load* — make sure the install cell ran (it needs `transformers`).
- *Slow first prediction* — the ViT weights download once on first load, then it's fast.